# Сбер: гиперперсонализированные рекомендации

Демо-notebook по мотивам проекта из презентации: строим рекомендательную модель для банковских продуктов на синтетических событиях клиентов.

Что внутри:
- synthetic client-product implicit feedback;
- matrix factorization через `TruncatedSVD`;
- offline-метрики `Recall@K` и `NDCG@K`;
- пример top-K рекомендаций для клиента.

Данные синтетические, чтобы notebook можно было запускать локально без приватных клиентских данных.

## Demo scope и production scale

Этот notebook — компактная демонстрация идеи, а не копия production-системы. Здесь всего 1200 синтетических клиентов, 12 продуктов и простая matrix factorization модель, чтобы пайплайн можно было быстро запустить локально.

В реальном проекте масштабы и архитектура были бы существенно крупнее:
- миллионы клиентов и миллиарды событий из транзакций, каналов коммуникации и поведения в приложении;
- feature store, ежедневные/почасовые витрины и скользящие окна истории;
- отдельные стадии candidate generation, ranking, business rules и post-filtering;
- PySpark/распределенное обучение, batch scoring и online inference;
- A/B-тесты, мониторинг CTR/conversion/uplift, latency, freshness и ошибок.

Смысл demo: показать end-to-end логику рекомендательной системы без приватных банковских данных и тяжелой инфраструктуры.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD

rng = np.random.default_rng(42)

N_CLIENTS = 1200
PRODUCTS = [
    "mortgage",
    "credit_card",
    "cash_loan",
    "investment_account",
    "savings_account",
    "premium_package",
    "travel_insurance",
    "auto_loan",
    "business_account",
    "salary_project",
    "kids_card",
    "brokerage",
]

client_ids = np.arange(N_CLIENTS)
segments = rng.choice(["young", "family", "business", "investor", "premium"], size=N_CLIENTS, p=[0.25, 0.25, 0.2, 0.2, 0.1])
segment_product_boost = {
    "young": ["credit_card", "kids_card", "travel_insurance"],
    "family": ["mortgage", "cash_loan", "savings_account", "kids_card"],
    "business": ["business_account", "salary_project", "premium_package"],
    "investor": ["investment_account", "brokerage", "savings_account"],
    "premium": ["premium_package", "brokerage", "travel_insurance"],
}

rows = []
for client_id, segment in zip(client_ids, segments):
    base = rng.uniform(0.02, 0.12, size=len(PRODUCTS))
    for product in segment_product_boost[segment]:
        base[PRODUCTS.index(product)] += rng.uniform(0.25, 0.5)
    probs = base / base.sum()
    n_events = int(rng.integers(3, 12))
    products = rng.choice(PRODUCTS, size=n_events, replace=True, p=probs)
    for t, product in enumerate(products):
        rows.append({"client_id": client_id, "segment": segment, "product": product, "event_time": t})

events = pd.DataFrame(rows)
events.head()

## Аналитический вывод

Синтетические данные задают разные клиентские сегменты и продуктовые склонности: например, семейные клиенты чаще взаимодействуют с ипотекой и накопительными продуктами, инвесторы — с брокерскими и инвестиционными продуктами. Это заменяет приватные банковские данные, но сохраняет смысл production-задачи: модель должна находить закономерности между поведением клиента и вероятностью интереса к продукту.

На реальном проекте вместо такой генерации использовались бы транзакции, поведение в приложении, история коммуникаций и продуктовый портфель клиента.

In [ ]:
# Последнее взаимодействие клиента оставляем в test, остальные используем для обучения.
events_sorted = events.sort_values(["client_id", "event_time"])
test = events_sorted.groupby("client_id").tail(1)
train = events_sorted.drop(test.index)

user_product = pd.crosstab(train["client_id"], train["product"]).reindex(index=client_ids, columns=PRODUCTS, fill_value=0)
interaction_matrix = user_product.to_numpy(dtype=float)

svd = TruncatedSVD(n_components=6, random_state=42)
user_factors = svd.fit_transform(interaction_matrix)
item_factors = svd.components_.T
scores = user_factors @ item_factors.T

# Уже купленные/показанные продукты не рекомендуем повторно.
scores[interaction_matrix > 0] = -np.inf
print(f"train events: {len(train):,}")
print(f"test events:  {len(test):,}")
print(f"matrix shape: {interaction_matrix.shape}")

## Аналитический вывод

Разбиение `last event holdout` имитирует задачу next-best-offer: по прошлой истории клиента нужно предсказать следующий потенциально интересный продукт. Матрица `client × product` компактно описывает implicit feedback, а `TruncatedSVD` выделяет скрытые факторы клиентов и продуктов.

Исключение уже увиденных продуктов важно для рекомендательной системы: без этого модель часто возвращала бы то, что клиент уже купил или уже видел, что снижает полезность рекомендаций.

In [ ]:
def recommend_for_client(client_id: int, k: int = 5) -> pd.DataFrame:
    row = scores[client_id]
    top_idx = np.argsort(-row)[:k]
    return pd.DataFrame({
        "rank": np.arange(1, k + 1),
        "client_id": client_id,
        "product": [PRODUCTS[i] for i in top_idx],
        "score": row[top_idx],
    })


def recall_at_k(k: int = 5) -> float:
    hits = 0
    total = 0
    truth = test.set_index("client_id")["product"].to_dict()
    for client_id, true_product in truth.items():
        recs = set(recommend_for_client(int(client_id), k)["product"])
        hits += int(true_product in recs)
        total += 1
    return hits / total


def ndcg_at_k(k: int = 5) -> float:
    values = []
    truth = test.set_index("client_id")["product"].to_dict()
    for client_id, true_product in truth.items():
        recs = recommend_for_client(int(client_id), k)["product"].tolist()
        if true_product in recs:
            rank = recs.index(true_product) + 1
            values.append(1.0 / np.log2(rank + 1))
        else:
            values.append(0.0)
    return float(np.mean(values))

for k in [3, 5, 10]:
    print(f"Recall@{k}: {recall_at_k(k):.3f} | NDCG@{k}: {ndcg_at_k(k):.3f}")

## Аналитический вывод

`Recall@K` показывает, как часто фактический следующий продукт клиента попадает в top-K рекомендаций. `NDCG@K` дополнительно учитывает позицию: попадание на первое место ценнее, чем попадание в конец списка.

Рост `Recall` при увеличении K ожидаем: чем больше рекомендаций показываем, тем выше шанс покрыть реальный интерес клиента. Для реального банка дальше нужно выбирать K не только по offline-метрикам, но и по UX, CTR, conversion и ограничениям каналов коммуникации.

In [ ]:
client_id = 42
print("segment:", segments[client_id])
print("history:")
display(train[train["client_id"] == client_id][["event_time", "product"]])

print("recommendations:")
display(recommend_for_client(client_id, k=5))

print("held-out next product:", test[test["client_id"] == client_id]["product"].iloc[0])

## Аналитический вывод

Для выбранного клиента видно, как модель использует историю взаимодействий и сегмент, чтобы предложить следующие продукты. Сравнение с held-out продуктом показывает локальный пример offline-проверки: если фактический следующий продукт попал в top-K, рекомендация считается успешной для `Recall@K`.

В production-проекте такой вывод можно использовать для объяснения рекомендаций бизнесу: какие продукты уже были в истории, что модель предлагает дальше и насколько это согласуется с последующим действием клиента.